In [19]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

# 1. Define the molecule using its SMILES string:
# 2-(4-(1,3-dihydro-2H-benzo[d]imidazol-2-ylidene)cyclohexa-2,5-dien-1-ylidene)malononitrile

FBz = "c1ccccc1F"
HFBz = "c1(c(c(c(c(c1F)F)F)F)F)F"
TFE = "C(C(F)(F)F)O"
HFIPA = "C(C(F)(F)F)(C(F)(F)F)O"
BrC18 = "CCCCCCCCCCCCCCCCCCBr"
phDADQ = "C1=CC=C2C(=C1)NC(=C3C=CC(=C(C#N)C#N)C=C3)N2"
CN6CP = "C1(=C(C#N)C#N)C(=C(C#N)C#N)C1=C(C#N)C#N"
Cl2 = "ClCl"
Br2 = "BrBr"
#new = "O1[Si]23O[Si]45O[Si]61O[Si]27O[Si]38O[Si]49O[Si]56O[Si]789"
new='[OH2]'
Molecules = [FBz, HFBz, TFE, HFIPA, BrC18, phDADQ, CN6CP, Cl2, Br2, new]


for i in Molecules:
    mol = Chem.MolFromSmiles(i)

    if mol is None:
        print("Error parsing the SMILES string.")
    else:
        # 2. Add hydrogens and generate initial 3D coordinates using ETKDG
        mol = Chem.AddHs(mol)
        AllChem.EmbedMolecule(mol, AllChem.ETKDG())
        
        # 3. Optimize geometry using the Universal Force Field (UFF)
        AllChem.UFFOptimizeMolecule(mol)

        # 4. Extract 3D atomic coordinates from the optimized conformer
        conf = mol.GetConformer()
        coords = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])

        # 5. Center coordinates at the origin (Mean = 0)
        coords_centered = coords - np.mean(coords, axis=0)

        # 6. Use Principal Component Analysis (Covariance matrix) to find the principal axes
        cov_matrix = np.cov(coords_centered, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        # Sort eigenvalues in descending order (Principal axis, Intermediate axis, Minor axis/Thickness)
        sort_idx = eigenvalues.argsort()[::-1]
        eigenvalues = eigenvalues[sort_idx]
        eigenvectors = eigenvectors[:, sort_idx]

        # Project atomic coordinates onto the principal axes to measure the exact span
        projections = np.dot(coords_centered, eigenvectors)
        min_proj = np.min(projections, axis=0)
        max_proj = np.max(projections, axis=0)
        
        # Raw nuclear spans along each axis (in Angstroms)
        raw_sizes = max_proj - min_proj

        # 7. Add Van der Waals margin (~1.5 Å per side per axis = ~3.0 Å total per dimension)
        vdw_buffer_angstroms = 3.0 
        dimensions_nm = (raw_sizes + vdw_buffer_angstroms) / 10.0

        print("--- Effective Molecular Dimensions (including VdW boundaries) ---")
        print(i)
        print(f"Principal Axis (Molecular Length):           {dimensions_nm[0]:.3f} nm")
        print(f"Intermediate Axis (Lateral Width/Bottleneck): {dimensions_nm[1]:.3f} nm")
        print(f"Minor Axis (Planar Thickness / Pi-stacking):  {dimensions_nm[2]:.3f} nm")
        print(f"Minimum diameter of encapsulation:"+ str(dimensions_nm[1]+0.34)+  " nm")
        print()
        print()

--- Effective Molecular Dimensions (including VdW boundaries) ---
c1ccccc1F
Principal Axis (Molecular Length):           0.824 nm
Intermediate Axis (Lateral Width/Bottleneck): 0.730 nm
Minor Axis (Planar Thickness / Pi-stacking):  0.300 nm
Minimum diameter of encapsulation:1.069983941994741 nm


--- Effective Molecular Dimensions (including VdW boundaries) ---
c1(c(c(c(c(c1F)F)F)F)F)F
Principal Axis (Molecular Length):           0.811 nm
Intermediate Axis (Lateral Width/Bottleneck): 0.846 nm
Minor Axis (Planar Thickness / Pi-stacking):  0.300 nm
Minimum diameter of encapsulation:1.1858311906867174 nm


--- Effective Molecular Dimensions (including VdW boundaries) ---
C(C(F)(F)F)O
Principal Axis (Molecular Length):           0.656 nm
Intermediate Axis (Lateral Width/Bottleneck): 0.576 nm
Minor Axis (Planar Thickness / Pi-stacking):  0.534 nm
Minimum diameter of encapsulation:0.9164622729458682 nm


--- Effective Molecular Dimensions (including VdW boundaries) ---
C(C(F)(F)F)(C(F)(F)F)O
